# **Calculo da taxa de toxicidade $k$ do artigo de Warner et al. (2022) [1]**

Este notebook calcula a taxa de toxicidade $k$ [$\text{min}^{-1}$] para cada composição de CPA presente nos dados experimentais, utilizando o modelo analítico reduzido de Warner et al. (2022) [1]. Os valores obtidos são salvos em `resultados_k_artigo.csv` e servem como referência para
a comparação com os valores estimados pela PINN no notebook principal.

In [7]:
import pandas as pd
import numpy as np
import itertools

## **Taxas de toxicidade (Warner) - Modelo Reduzido** 

### **O modelo reduzido**

O modelo decompõe $k$ em três termos aditivos: toxicidade específica individual (proporcional a $M_i^{\alpha_i}$ para cada CPA livre), toxicidade específica binária (proporcional a $M_{ij}^{\alpha_{ij}}$ para cada par em complexo) e toxicidade não-específica (potência $\alpha_n$ da soma ponderada de todas as concentrações livres e complexadas). 

No modelo reduzido, os termos de toxicidade específica binária são nulos para todos os pares ($\zeta_{ij} = 0$), de modo que a interação entre componentes entra exclusivamente pelo termo não-específico via $\beta_{ij}$.

### **O cálculo**

Antes de avaliar os três termos do modelo, é necessário determinar quanto de cada CPA está "livre" em solução — pois parte pode estar associada em complexos binários com constante de equilíbrio $K_{eq} > 0$. Para cada par $(i, j)$ com $K_{eq} > 0$, a concentração do complexo $M_{ij}$ é obtida analiticamente pela fórmula quadrática derivada do equilíbrio $K_{eq} = M_{ij} / (M_i \cdot M_j)$, resolvida em termos das concentrações totais $M_i^T$ e $M_j^T$:

$$M_{ij} = \frac{(K M_i^T + K M_j^T + 1) - \sqrt{(K M_i^T + K M_j^T + 1)^2 - 4K^2 M_i^T M_j^T}}{2K}$$

A concentração livre de cada componente é então $M_{i,\text{sol}} = M_i^T - \sum_j M_{ij}$, com um *clipping* (um "encaixe") em zero para evitar valores negativos decorrentes da acumulação de descontos sequenciais entre pares. Os pares sem complexação ($K_{eq} = 0$) contribuem com $M_{ij} = 0$ e não alteram as concentrações livres.

A fórmula resultante para o modelo é a seguinte:

$$
k=\sum_{i=1}^{n}\zeta_{i}M_{i_{sol}}^{\alpha_{i}}+\sum_{i=1}^{n-1}\sum_{j=i+1}^{n}\zeta_{ij}M_{ij}^{\alpha_{ij}}+(\sum_{i=1}^{n}\beta_{i}M_{i_{sol}}+\sum_{i=1}^{n-1}\sum_{j=i+1}^{n}\beta_{ij}M_{ij})^{\alpha_{n}}
$$

### **Execução**

Os .csv gerados pelo notebook de extração (``dados_simples_e_binarios.csv`` e ``dados_ternarios.csv``) são concatenados e as linhas duplicadas removidas, de forma que $k$ é calculado uma única vez por composição, e não uma vez por réplica ou por ponto temporal. O resultado é um *DataFrame* de 86 composições únicas com a coluna ``k_artigo``, salvo em ``resultados_k_artigo.csv``.

Definição dos valores das variáveis obtidos pelo modelo não-linear encontrados na tabela suplementar do artigo:

In [8]:
alpha_n = 9.51

zeta = {"Gly": 2.95e-4, "DMSO": 6.54e-4, "PG": 0.0, "EG": 0.0, "FA": 8.85e-4}
alpha = {"Gly": 1.75, "DMSO": 1.27e-3, "PG": 0.0, "EG": 0.0, "FA": 2.87}
beta = {"Gly": 6.70e-2, "DMSO": 8.32e-2, "PG": 1.28e-1, "EG": 6.98e-2, "FA": 1.06e-1}

zeta_ij = {
    ("Gly", "DMSO"): 0.0, ("Gly", "PG"): 0.0, ("Gly", "EG"): 0.0, ("Gly", "FA"): 0.0,
    ("DMSO", "PG"): 0.0, ("DMSO", "EG"): 0.0, ("DMSO", "FA"): 0.0,
    ("PG", "EG"): 0.0, ("PG", "FA"): 0.0,
    ("EG", "FA"): 0.0
}

alpha_ij = {
    ("Gly", "DMSO"): 0.0, ("Gly", "PG"): 0.0, ("Gly", "EG"): 0.0, ("Gly", "FA"): 0.0,
    ("DMSO", "PG"): 0.0, ("DMSO", "EG"): 0.0, ("DMSO", "FA"): 0.0,
    ("PG", "EG"): 0.0, ("PG", "FA"): 0.0,
    ("EG", "FA"): 0.0
}

beta_ij = {
    ("Gly", "DMSO"): 7.99e-2, ("Gly", "PG"): 1.07e-2, ("Gly", "EG"): 3.42e-8, ("Gly", "FA"): 9.33e-4,
    ("DMSO", "PG"): 9.24e-2, ("DMSO", "EG"): 5.66e-9, ("DMSO", "FA"): 6.96e-2,
    ("PG", "EG"): 1.51e-1, ("PG", "FA"): 7.37e-4,
    ("EG", "FA"): 1.27e-1
}

K_eq = {
    ("Gly", "DMSO"): 1.26e-2, ("Gly", "PG"): 0.0, ("Gly", "EG"): 1.31e-2, ("Gly", "FA"): 1.29e-1,
    ("DMSO", "PG"): 3.21e-2, ("DMSO", "EG"): 0.0, ("DMSO", "FA"): 8.59e-2,
    ("PG", "EG"): 1.63e-1, ("PG", "FA"): 0.0,
    ("EG", "FA"): 5.93e-2
}

COMBINACOES_IJ = [
    ("Gly", "DMSO"), ("Gly", "PG"), ("Gly", "EG"), 
    ("Gly", "FA"), ("DMSO", "PG"), ("DMSO", "EG"),
    ("DMSO", "FA"), ("PG", "EG"), ("PG", "FA"),
    ("EG", "FA")
]

A função que itera sobre o dataset e calcula por aquela grande fórmula o valor de $k$.

In [9]:
def calcular_k_warner(linha):
    M_total = {
        "Gly": linha[1],
        "DMSO": linha[2],
        "PG": linha[3],
        "EG": linha[4],
        "FA": linha[5]
    }
    cpas = list(M_total.keys())
    M_solto = M_total.copy()

    M_ij = {}

    for i, j in COMBINACOES_IJ:
        par = (i, j)
        K = K_eq.get(par)

        if K > 0:
            Mi_T = M_total[i]
            Mj_T = M_total[j]

            termo_b = (K * Mi_T) + (K * Mj_T) + 1

            termo_radicando = (K**2 * Mi_T**2) + (K**2 * Mj_T**2) - (2 * K**2 * Mi_T * Mj_T) + (2 * K * Mi_T) + (2 * K * Mj_T) + 1

            M_ij[par] = (termo_b - np.sqrt(termo_radicando)) / (2 * K)
        
        else:
            M_ij[par] = 0
        
        M_solto[i] -= M_ij[par]
        M_solto[j] -= M_ij[par]

        if M_solto[i] < 0:
            M_solto[i] = 0.0
            # print(f"M_solto de {i} foi menor que 0! - Valor: {M_solto[i]}")
        
        if M_solto[j] < 0:
            M_solto[j] = 0.0
            # print(f"{j} foi menor que 0! - Valor: {M_solto[j]}")
        
    termo_tox_esp_ind = sum(zeta[i] * (M_solto[i] ** alpha[i]) for i in cpas)

    termo_tox_esp_bin = sum(zeta_ij[par] * (M_ij[par] ** alpha_ij[par]) for par in M_ij)

    soma_beta_ind = sum(beta[i] * M_solto[i] for i in cpas)
    soma_beta_bin = sum(beta_ij[par] * M_ij[par] for par in M_ij)
    termo_tox_nao_esp = (soma_beta_ind + soma_beta_bin) ** alpha_n
    
    
    k_final = termo_tox_esp_ind + termo_tox_esp_bin + termo_tox_nao_esp
    return k_final

Parte da concatenação dos DataFrames e regristo no *.csv* dos valores de $k$.

In [10]:
df_simp_bin = pd.read_csv("dados/dados_simples_e_binarios.csv")
df_ter = pd.read_csv("dados/dados_ternarios.csv")

df_cpas= pd.concat([df_simp_bin, df_ter], ignore_index=True)

colunas_cpas = ["Gly (mol/L)", "DMSO (mol/L)", "PG (mol/L)", "EG (mol/L)", "FA (mol/L)"]

df_cpas_unicos = df_cpas[colunas_cpas].drop_duplicates().reset_index(drop=True)

k_vals = []
for linha in df_cpas_unicos.itertuples():
    k = calcular_k_warner(linha)
    k_vals.append(k)

print(k_vals)
df_cpas_unicos["k_artigo"] = k_vals

print(f"Foram calculados os K de {len(df_cpas_unicos)} misturas únicas.")
display(df_cpas_unicos.head())

[np.float64(0.0), np.float64(0.0002950000068544261), np.float64(0.0020176000077477313), np.float64(0.004962380933343006), np.float64(0.009632960155617418), np.float64(0.03939723627857152), np.float64(0.0006540000537493025), np.float64(0.000656765783283639), np.float64(0.0008938871197187524), np.float64(0.0065069563391434646), np.float64(0.1745852488950309), np.float64(3.232710481515111e-09), np.float64(0.00011142687319488207), np.float64(0.014974265410912294), np.float64(0.3525511523380206), np.float64(10.460846065082137), np.float64(1.0117310277783977e-11), np.float64(3.487291100895736e-07), np.float64(4.490242647238422e-05), np.float64(0.0011014059788874756), np.float64(0.03273897431080086), np.float64(0.0008850005378470349), np.float64(0.0207333639573397), np.float64(0.09212729202007125), np.float64(0.29426009491929406), np.float64(2.3964984521701824), np.float64(0.0007401706914917097), np.float64(0.001235741924897265), np.float64(0.0021246108968478034), np.float64(0.004952990975839

,Gly (mol/L),DMSO (mol/L),PG (mol/L),EG (mol/L),FA (mol/L),k_artigo
0,0.0,0.0,0.0,0.0,0.0,0.000000
1,1.0,0.0,0.0,0.0,0.0,0.000295
2,3.0,0.0,0.0,0.0,0.0,0.002018
3,5.0,0.0,0.0,0.0,0.0,0.004962
4,7.0,0.0,0.0,0.0,0.0,0.009633


Exportação do DataFrame para uma tabela *.csv*

In [11]:
df_cpas_unicos.to_csv('resultados_k_artigo.csv', index=False)

## **Referências**

[1] Warner, R. M., et al. (2022). Rapid Quantification of Cryoprotectant Permeability via Numerical Optimization of Differential Scanning Calorimetry Thermograms. Cryobiology. DOI: 10.1016.

**Autor**: Arthur Brandão do Nascimento